In [10]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import importlib, io
import pandas as pd
import pipeline.bronze, pipeline.convert, pipeline.silver

importlib.reload(pipeline.bronze)
importlib.reload(pipeline.convert)
importlib.reload(pipeline.silver)

from pipeline.convert import get_parquet
from pipeline.silver import map_columns, load_schema

In [11]:
retail_cols = list(pd.read_parquet(io.BytesIO(get_parquet("online_retail_ii"))).columns)
print(retail_cols)
print(list(load_schema()))

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
['invoice_id', 'product_code', 'product_name', 'quantity', 'order_timestamp', 'unit_price', 'customer_id', 'country']


In [13]:
importlib.reload(pipeline.silver)
from pipeline.silver import map_columns

expected = {
    "Invoice": "invoice_id", "StockCode": "product_code",
    "Description": "product_name", "Quantity": "quantity",
    "InvoiceDate": "order_timestamp", "Price": "unit_price",
    "Customer ID": "customer_id", "Country": "country",
}


# 1. Kontrollkällan ska vara oförändrad: allt mappas exakt
result = map_columns(retail_cols)
assert result["mapped"] == expected
assert result["suggested"] == {} and result["flagged"] == [] and result["missing"] == []

# 2. Schema drift: omdöpt kolumn ska föreslås, ny okänd ska flaggas
drifted = [("price_sek" if c == "Price" else c) for c in retail_cols] + ["Discount"]
result = map_columns(drifted)

assert list(result["suggested"]) == ["price_sek"]
assert result["suggested"]["price_sek"]["target"] == "unit_price"
assert result["flagged"] == ["Discount"]
assert result["missing"] == ["unit_price"]   # saknas tills förslaget godkänts
n = len(result["mapped"]) + len(result["suggested"]) + len(result["flagged"])
assert n == len(drifted), "En kolumn försvann"

# 3. Kolumner som liknar men betyder något annat ska INTE föreslås
result = map_columns(["Customer Email", "Unit Cost", "Warehouse"])
assert result["suggested"] == {}

print(map_columns(drifted))

{'mapped': {'Invoice': 'invoice_id', 'StockCode': 'product_code', 'Description': 'product_name', 'Quantity': 'quantity', 'InvoiceDate': 'order_timestamp', 'Customer ID': 'customer_id', 'Country': 'country'}, 'suggested': {'price_sek': {'target': 'unit_price', 'matched_on': 'price', 'score': 90.0}}, 'flagged': ['Discount'], 'missing': ['unit_price']}
